In [ ]:
from pathlib import Path
from utils.model import model_gemini_retry
import logfire
from pydantic_ai import Agent
from dotenv import load_dotenv
from pydantic_ai_harness.subagents import SubAgent, SubAgents
from pydantic_ai_harness.planning import Planning
from pydantic_ai.common_tools.tavily import tavily_search_tool
load_dotenv()
from utils.output_schema import TravelPlan

api_key = "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"

PROMPTS_DIR = Path("prompts")


def load_instructions(name: str) -> str:
    """Read an agent's instructions from prompts/<name>.j2."""
    return (PROMPTS_DIR / f"{name}.j2").read_text()

In [2]:
logfire.configure()
logfire.instrument_pydantic_ai()

Logfire project URL: https://logfire-eu.pydantic.dev/johanidler/rain

In [ ]:
create_booking_agent = Agent(
    model='openai:gpt-5.6-luna',
    name="create_booking_agent",
    description="Provide this Agent with exact information, it will then navigate the webstie to create the booking",
    instructions=load_instructions("create_booking_agent"),
    # this one needs a way to access websites and pay for them
)

reserach_hotels_agent = Agent(
    model='openai:gpt-5.6-luna',
    name="reserach_hotels_agent",
    description="Researches hotel options for a destination and reports back a short list, without booking",
    instructions=load_instructions("reserach_hotels_agent"),
    tools=[tavily_search_tool(api_key)]
)

research_flights_agent = Agent(
    model='openai:gpt-5.6-luna',
    name="research_flights_agent",
    description="Researches flight options for a route and dates and reports back a short list, without booking",
    instructions=load_instructions("research_flights_agent"),
    tools=[tavily_search_tool(api_key)]
)

In [ ]:
travel_agent = Agent(
    model='openai:gpt-5.6-luna',
    instructions=load_instructions("travel_agent"),
    output_type=TravelPlan,
    capabilities=[
        Planning(inject=False),
        SubAgents(agents=[
            SubAgent(create_booking_agent),
            SubAgent(reserach_hotels_agent),
            SubAgent(research_flights_agent),
        ])
    ],

)

In [ ]:
trip_request = Path("data01.json").read_text()
res = await travel_agent.run(trip_request)
print(res.output.model_dump_json(indent=2))

In [ ]:
with open('output.json', 'w') as f:
    f.write(res.output.model_dump_json(indent=2))